# frequency_test — MFT / FMFT / FMFT2 frequency analysis

Runs REBOUND's frequency analysis in all three modes on a synthetic three-frequency signal (true frequencies 0.30, 0.55, 0.11 radians per sample) and prints the recovered frequencies, amplitudes and phases.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example frequency_test
cd porttest
../target/release/examples/frequency_test
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "frequency_test"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

frequency_test done



In [4]:
p = os.path.join(WORK, "frequency_rust.txt")
vals, mode = [], None
for line in open(p).read().splitlines():
    parts = line.split()
    if len(parts) == 3 and parts[1] == "ret":
        mode, vals = parts[0], []
        print(f"--- {mode} (ret {parts[2]}) ---")
    elif len(parts) == 2:
        vals.append(unbits(parts[1]))
        if len(vals) == 9:
            print("  frequencies:", [round(v, 6) for v in vals[0:3]])
            print("  amplitudes :", [round(v, 6) for v in vals[3:6]])
            print("  phases     :", [round(v, 6) for v in vals[6:9]])


--- MFT (ret 0) ---
  frequencies: [0.299996, 0.550001, 0.11]
  amplitudes : [1.000062, 0.349998, 0.1]
  phases     : [0.400489, 1.899897, 5.099987]
--- FMFT (ret 0) ---
  frequencies: [0.3, 0.55, 0.11]
  amplitudes : [1.0, 0.35, 0.1]
  phases     : [0.4, 1.9, 5.1]
--- FMFT2 (ret 0) ---
  frequencies: [0.3, 0.55, 0.11]
  amplitudes : [1.0, 0.35, 0.1]
  phases     : [0.4, 1.9, 5.1]
